In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os
import csv
import torch

c:\Users\joaop\miniconda3\envs\AP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

print("CUDA está disponível?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nome da Placa Gráfica:", torch.cuda.get_device_name(0))

CUDA está disponível? True
Nome da Placa Gráfica: NVIDIA GeForce GTX 960M


In [3]:
print("🚀 A iniciar a preparação dos dados do MindGuard AI...")

DIRETORIO_ATUAL = os.getcwd() 
FICHEIRO_ENTRADA = os.path.join(DIRETORIO_ATUAL, "dataset_treino_raw.csv")

df = pd.read_csv(FICHEIRO_ENTRADA)

🚀 A iniciar a preparação dos dados do MindGuard AI...


In [4]:
# 2. O Modelo precisa de números, não de texto para as Labels
labels = df['Contexto_Label'].unique().tolist()
label2id = {lbl: i for i, lbl in enumerate(labels)}
id2label = {i: lbl for i, lbl in enumerate(labels)}

df['label_num'] = df['Contexto_Label'].map(label2id)
df = df.rename(columns={'label_num': 'labels'})

In [5]:
# 3. Dividir os dados: 80% treino, 20% teste
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
ds_train = Dataset.from_pandas(train_df)
ds_test = Dataset.from_pandas(test_df)

In [6]:
# 4. Descarregar o modelo e tokenizer
modelo_nome = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(modelo_nome)

def tokenizar_frases(exemplos):
    return tokenizer(exemplos["Acao"], padding="max_length", truncation=True, max_length=128)

ds_train = ds_train.map(tokenizar_frases, batched=True)
ds_test = ds_test.map(tokenizar_frases, batched=True)

modelo = AutoModelForSequenceClassification.from_pretrained(
    modelo_nome,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 10378.16it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier

In [7]:
# --- NOVO: Ensinar o modelo a calcular a Accuracy ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

In [9]:
# 5. Configurar o Treino
args = TrainingArguments(
    output_dir="./resultados_mindguard",
    eval_strategy="epoch",      # Avalia no fim de cada ciclo
    logging_strategy="epoch",   # <-- NOVO: Obriga a guardar o loss do treino a cada ciclo para o gráfico
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,         
    weight_decay=0.01,
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics # <-- NOVO: Injeta a função de accuracy
)

In [10]:
# 6. INICIAR A MAGIA
print(f"🧠 O DeBERTa está a aprender as tuas {len(labels)} categorias. A treinar...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


🧠 O DeBERTa está a aprender as tuas 4 categorias. A treinar...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# 7. Guardar o modelo final
trainer.save_model("./modelo_mindguard_final")
tokenizer.save_pretrained("./modelo_mindguard_final")
print("✅ Treino concluído com sucesso! A Inteligência Artificial está pronta.\n")

In [ ]:
# ==========================================
# 8. EXTRAIR HISTÓRICO E GERAR GRÁFICOS
# ==========================================
print("📊 A processar a telemetria e a gerar os gráficos...")

# O Trainer guarda tudo numa lista de dicionários confusa. Vamos organizá-la por epoch!
historico_limpo = {}
for log in trainer.state.log_history:
    epoch = round(log.get('epoch', 0), 2)
    if epoch not in historico_limpo:
        historico_limpo[epoch] = {}
    historico_limpo[epoch].update(log)

# Preparar as listas para o plot
epochs_list, train_loss, val_loss, val_acc = [], [], [], []

for ep in sorted(historico_limpo.keys()):
    dados = historico_limpo[ep]
    # Só adicionamos se tivermos a loss de treino E a loss de validação nessa época
    if 'loss' in dados and 'eval_loss' in dados:
        epochs_list.append(ep)
        train_loss.append(dados['loss'])
        val_loss.append(dados['eval_loss'])
        val_acc.append(dados.get('eval_accuracy', 0)) # Captura a accuracy que configurámos!

# Desenhar com Matplotlib e Seaborn
sns.set_style('whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico 1: Accuracy (Só temos Validação porque o HF não calcula accuracy no treino nativamente)
ax1.plot(epochs_list, val_acc, label='validation', color='#ff7f0e', linewidth=2, marker='o')
ax1.set_title('Accuracy ao longo das Épocas', fontsize=14)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.legend(loc='lower right')

# Gráfico 2: Loss (Treino vs Validação)
ax2.plot(epochs_list, train_loss, label='training loss', color='#1f77b4', linewidth=2, marker='o')
ax2.plot(epochs_list, val_loss, label='validation loss', color='#ff7f0e', linewidth=2, marker='o')
ax2.set_title('Loss (Erro) ao longo das Épocas', fontsize=14)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()